In [2]:
# Data manipulation
import pandas as pd
import numpy as np

# Machine Learning Models
from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# Deep Learning with xrnn
from xrnn.model import Model
from xrnn.layers import Conv2D, BatchNormalization, Flatten, Dense, Dropout, MaxPooling2D
from xrnn.activations import ReLU, Sigmoid
from xrnn.losses import BinaryCrossentropy
from xrnn.optimizers import Adam

# Utilities
import joblib
import warnings
import time
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print(" Libraries imported successfully!")

 Libraries imported successfully!


In [3]:
print("="*60)
print(" LOADING PREPROCESSED DATA")
print("="*60)

X_train = pd.read_csv('../data/processed/UNSW_X_train_preprocessed.csv')
X_val = pd.read_csv('../data/processed/UNSW_X_val_preprocessed.csv')
X_test = pd.read_csv('../data/processed/UNSW_X_test_preprocessed.csv')

y_train = pd.read_csv('../data/processed/UNSW_y_train.csv').values.ravel()
y_val = pd.read_csv('../data/processed/UNSW_y_val.csv').values.ravel()
y_test = pd.read_csv('../data/processed/UNSW_y_test.csv').values.ravel()

class_weights = joblib.load('../models/UNSW/class_weights.pkl')

print(f"Training: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")
print(f"Class weights: {class_weights}")

 LOADING PREPROCESSED DATA
Training: (122738, 34)
Validation: (52603, 34)
Test: (82332, 34)
Class weights: {0: np.float64(1.5655357142857143), 1: np.float64(0.7346237640355288)}


In [4]:
print("="*60)
print(" LINEAR SVC BASELINE (For Comparison)")
print("="*60)

# Train a fast LinearSVC as baseline
print("Training LinearSVC baseline...")
start = time.time()

linear_baseline = LinearSVC(
    random_state=42,
    max_iter=2000,
    class_weight='balanced',
    dual='auto',
    loss='squared_hinge',
    tol=0.0001
)

linear_baseline.fit(X_train, y_train)
train_time = time.time() - start

# Predict on validation
y_pred_linear = linear_baseline.predict(X_val)

# Calculate metrics
linear_recall = recall_score(y_val, y_pred_linear)
linear_f1 = f1_score(y_val, y_pred_linear)
linear_accuracy = accuracy_score(y_val, y_pred_linear)

print(f"\n LinearSVC Training Time: {train_time:.2f}s")
print("-" * 40)
print(" LinearSVC Performance (Validation Set):")
print(f"  Accuracy: {linear_accuracy:.4f}")
print(f"  Recall:   {linear_recall:.4f}")
print(f"  F1-Score: {linear_f1:.4f}")
print("-" * 40)

# Store for later comparison
baseline_results = {
    'Model': 'LinearSVC (Baseline)',
    'Accuracy': linear_accuracy,
    'Recall': linear_recall,
    'F1-Score': linear_f1,
    'Time': f"{train_time:.2f}s"
}

print("\n This baseline will be compared against RBF SVM and CNN later.")

 LINEAR SVC BASELINE (For Comparison)
Training LinearSVC baseline...

 LinearSVC Training Time: 6.14s
----------------------------------------
 LinearSVC Performance (Validation Set):
  Accuracy: 0.8701
  Recall:   0.8737
  F1-Score: 0.9015
----------------------------------------

 This baseline will be compared against RBF SVM and CNN later.


In [5]:
print("="*60)
print(" RBF SVM HYPERPARAMETER TUNING")
print("="*60)

# Use 20% of training data for tuning (still 35k samples - plenty for parameter selection)
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train, 
    train_size=0.2,
    random_state=42,
    stratify=y_train
)
print(f"Tuning data: {X_tune.shape} ({len(X_tune)} samples)")

# Focused parameter grid based on research recommendations
param_grid = {
    'C': [0.1, 1, 10],           # Regularization
    'gamma': ['scale', 0.1, 0.01], # RBF specific parameter
    'kernel': ['rbf'],
    'class_weight': ['balanced']
}

print("\n RBF Parameter grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")
print(f"Total combinations: 3 × 3 = 9 combinations")

# Use RandomizedSearchCV with 5-fold CV (but fewer iterations)
print("\n Performing randomized search (will take 5-8 minutes)...")

# CRITICAL: probability=False for tuning (much faster)
rbf_svm = SVC(probability=False, random_state=42, cache_size=1000)

random_search = RandomizedSearchCV(
    rbf_svm,
    param_distributions=param_grid,
    n_iter=9,  # Try all 9 combinations
    cv=3,      # 3-fold CV
    scoring='recall',
    verbose=1,
    n_jobs=-1,
    random_state=42
)

start_time = time.time()
random_search.fit(X_tune, y_tune)
tuning_time = time.time() - start_time

print(f"\n Tuning complete! Time: {tuning_time:.2f}s ({tuning_time/60:.1f} minutes)")
print(f"\nBest parameters found:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"Best cross-validation recall: {random_search.best_score_:.4f}")

best_rbf_params = random_search.best_params_
joblib.dump(best_rbf_params, '../models/UNSW/best_rbf_params.pkl')

 RBF SVM HYPERPARAMETER TUNING
Tuning data: (24547, 34) (24547 samples)

 RBF Parameter grid:
  C: [0.1, 1, 10]
  gamma: ['scale', 0.1, 0.01]
  kernel: ['rbf']
  class_weight: ['balanced']
Total combinations: 3 × 3 = 9 combinations

 Performing randomized search (will take 5-8 minutes)...
Fitting 3 folds for each of 9 candidates, totalling 27 fits

 Tuning complete! Time: 395.98s (6.6 minutes)

Best parameters found:
  kernel: rbf
  gamma: 0.01
  class_weight: balanced
  C: 1
Best cross-validation recall: 0.9510


['../models/UNSW/best_rbf_params.pkl']

In [ ]:
print("="*60)
print(" TRAINING FINAL RBF SVM")
print("="*60)

best_params = joblib.load('../models/UNSW/best_rbf_params.pkl')
print(f"Training with params: {best_params}")

# Add probability=True only for final model (single training)
final_params = best_params.copy()
final_params['probability'] = True
final_params['cache_size'] = 2000  # Increase cache for speed

print("\n Training on full dataset.")
start = time.time()

final_svm = SVC(**final_params, random_state=42)
final_svm.fit(X_train, y_train)
train_time = time.time() - start

print(f" Training complete! Time: {train_time:.2f}s ({train_time/60:.1f} minutes)")

# Predict on validation
y_pred_val = final_svm.predict(X_val)
val_recall = recall_score(y_val, y_pred_val)
val_f1 = f1_score(y_val, y_pred_val)

print(f"\n Validation Performance:")
print(f"  Recall: {val_recall:.4f}")
print(f"  F1-Score: {val_f1:.4f}")

joblib.dump(final_svm, '../models/UNSW/final_rbf_svm.pkl')
print(" RBF SVM model saved")

 TRAINING FINAL RBF SVM
Training with params: {'kernel': 'rbf', 'gamma': 0.01, 'class_weight': 'balanced', 'C': 1}

 Training on full dataset.


In [5]:
# Diagnostic: Check current training progress
import psutil
import os

print("="*60)
print("TRAINING DIAGNOSTIC")
print("="*60)

# Check system resources
print(f"CPU Usage: {psutil.cpu_percent()}%")
print(f"Memory Usage: {psutil.virtual_memory().percent}%")
print(f"Available Memory: {psutil.virtual_memory().available / 1e9:.2f} GB")

# Check if training is actually progressing
if hasattr(final_svm, 'n_features_in_'):
    print("\n Model is partially trained")
else:
    print("\n Model may be stuck")

TRAINING DIAGNOSTIC
CPU Usage: 43.8%
Memory Usage: 62.3%
Available Memory: 6.41 GB


NameError: name 'final_svm' is not defined

In [6]:
print("="*60)
print("PREPARING DATA FOR CNN")
print("="*60)

# Convert to numpy arrays
X_train_np = X_train.values.astype(np.float32)
X_val_np = X_val.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)
y_train_np = y_train.astype(np.float32)
y_val_np = y_val.astype(np.float32)
y_test_np = y_test.astype(np.float32)

# Reshape for Conv2D: 34 features → 17x2 grid
X_train_cnn = X_train_np.reshape(-1, 17, 2, 1)
X_val_cnn = X_val_np.reshape(-1, 17, 2, 1)
X_test_cnn = X_test_np.reshape(-1, 17, 2, 1)

print(f" CNN input shape: {X_train_cnn.shape}")
print(f"   (samples, height=17, width=2, channels=1)")

PREPARING DATA FOR CNN
 CNN input shape: (122738, 17, 2, 1)
   (samples, height=17, width=2, channels=1)


In [7]:
print("="*60)
print(" BUILDING CNN MODEL (xrNN Correct API)")
print("="*60)

from xrnn.activations import ReLU, Sigmoid

def create_cnn():
    model = Model()
    
    # First convolutional block
    model.add(Conv2D(32, kernel_size=(3, 1), padding='valid'))
    model.add(ReLU())
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 1)))
    model.add(Dropout(0.25))
    
    # Second convolutional block
    model.add(Conv2D(64, kernel_size=(3, 1), padding='valid'))
    model.add(ReLU())
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 1)))
    model.add(Dropout(0.25))
    
    # Third convolutional block
    model.add(Conv2D(128, kernel_size=(3, 1), padding='valid'))
    model.add(ReLU())
    model.add(BatchNormalization())
    model.add(Dropout(0.25))
    
    # Flatten
    model.add(Flatten())
    
    # Dense layers
    model.add(Dense(256))
    model.add(ReLU())
    model.add(Dropout(0.3))
    
    model.add(Dense(128))
    model.add(ReLU())
    model.add(Dropout(0.2))
    
    # Output
    model.add(Dense(1))
    model.add(Sigmoid())
    
    return model

# Create model
print("Creating CNN...")
cnn_model = create_cnn()

# Set optimizer and loss as attributes (with correct parameter name)
from xrnn.optimizers import Adam
from xrnn.losses import BinaryCrossentropy

# Fix: Use 'learning_rate' instead of 'lr'
cnn_model.optimizer = Adam(learning_rate=0.001)
cnn_model.loss = BinaryCrossentropy()

print(" CNN built and configured successfully!")
print("\n Architecture:")
print("   Conv2D(32) → ReLU → BatchNorm → MaxPool → Dropout")
print("   Conv2D(64) → ReLU → BatchNorm → MaxPool → Dropout")
print("   Conv2D(128) → ReLU → BatchNorm → Dropout")
print("   Flatten → Dense(256) → ReLU → Dropout")
print("   Dense(128) → ReLU → Dropout → Dense(1) → Sigmoid")

 BUILDING CNN MODEL (xrNN Correct API)
Creating CNN...
 CNN built and configured successfully!

 Architecture:
   Conv2D(32) → ReLU → BatchNorm → MaxPool → Dropout
   Conv2D(64) → ReLU → BatchNorm → MaxPool → Dropout
   Conv2D(128) → ReLU → BatchNorm → Dropout
   Flatten → Dense(256) → ReLU → Dropout
   Dense(128) → ReLU → Dropout → Dense(1) → Sigmoid


In [20]:
import inspect
print("Train method parameters:")
print(inspect.signature(cnn_model.train))

Train method parameters:
(x: numpy.ndarray | list | xrnn.data_handler.SupportsGetitem, y: numpy.ndarray | list = None, batch_size: int = 32, epochs: int = 1, shuffle: bool = True, validation_data: List[list | numpy.ndarray] | xrnn.data_handler.SupportsGetitem = None, validation_split: float = 0.0, validation_freq: int = 1, steps_per_epoch: int = None, print_every: int = 1, disable_sleep: bool = False) -> None


In [8]:
print("="*60)
print(" XRNN CNN TRAINING WITH FULL SAVES")
print("="*60)

import time
import pickle
import joblib
import os
from sklearn.metrics import recall_score, f1_score, accuracy_score

# Create save directories if they don't exist
os.makedirs('../models/UNSW', exist_ok=True)
os.makedirs('../results/UNSW', exist_ok=True)

# Save model BEFORE training (backup)
joblib.dump(cnn_model, '../models/UNSW/xrnn_model_initial.pkl')
print(" Initial model saved to models/UNSW/xrnn_model_initial.pkl")

print("\nStarting training...")
start = time.time()

# Train the model
history = cnn_model.train(
    X_train_cnn, 
    y_train_np,
    batch_size=256,
    epochs=30,
    shuffle=True,
    validation_data=(X_val_cnn, y_val_np),
    validation_freq=1,
    print_every=1,
    disable_sleep=False
)

# Save checkpoint every 5 epochs (if you can access epoch number)
# Note: This depends on if history contains epoch info
try:
    for epoch in range(5, 31, 5):
        checkpoint_path = f'../models/UNSW/xrnn_model_epoch{epoch}.pkl'
        joblib.dump(cnn_model, checkpoint_path)
        print(f" Checkpoint saved at epoch {epoch}")
except:
    print(" Could not save checkpoints during training")

train_time = time.time() - start
print(f"\n CNN training complete! Time: {train_time:.2f}s ({train_time/60:.1f} minutes)")

# Save final model
joblib.dump(cnn_model, '../models/UNSW/xrnn_model_final.pkl')
print(" Final model saved to: ../models/UNSW/xrnn_model_final.pkl")

# Evaluate on validation set
print("\n Evaluating on validation set...")
y_pred_cnn = cnn_model.predict(X_val_cnn)
y_pred_cnn_binary = (y_pred_cnn > 0.5).astype(int).flatten()

# Calculate metrics
cnn_recall = recall_score(y_val, y_pred_cnn_binary)
cnn_f1 = f1_score(y_val, y_pred_cnn_binary)
cnn_accuracy = accuracy_score(y_val, y_pred_cnn_binary)

print(f"\n CNN Validation Performance:")
print(f"  Accuracy: {cnn_accuracy:.4f}")
print(f"  Recall:   {cnn_recall:.4f}")
print(f"  F1-Score: {cnn_f1:.4f}")

# Save validation results
val_results = {
    'accuracy': float(cnn_accuracy),
    'recall': float(cnn_recall),
    'f1_score': float(cnn_f1),
    'training_time_minutes': train_time/60
}

with open('../results/UNSW/xrnn_validation_results.json', 'w') as f:
    import json
    json.dump(val_results, f, indent=4)
print(" Validation results saved to: ../results/UNSW/xrnn_validation_results.json")

# Save training history
with open('../results/UNSW/xrnn_history.pkl', 'wb') as f:
    pickle.dump(history, f)
print(" Training history saved to: ../results/UNSW/xrnn_history.pkl")

print("\n" + "="*60)
print(" ALL FILES SAVED SUCCESSFULLY")
print("="*60)
print(" Models saved:")
print("  - ../models/UNSW/xrnn_model_initial.pkl (backup)")
print("  - ../models/UNSW/xrnn_model_final.pkl (final)")
print(" Results saved:")
print("  - ../results/UNSW/xrnn_validation_results.json")
print("  - ../results/UNSW/xrnn_history.pkl")

 XRNN CNN TRAINING WITH FULL SAVES
 Initial model saved to models/UNSW/xrnn_model_initial.pkl

Starting training...
Epoch 1/30
step: 480/480 [==============================] - 100% - Took: 7.5 min - loss: 0.678 - acc: 0.658 - val_loss: 0.632 - val_acc: 0.681

Epoch 2/30
step: 480/480 [==============================] - 100% - Took: 10.0 min - loss: 0.632 - acc: 0.680 - val_loss: 0.631 - val_acc: 0.681

Epoch 3/30
step: 480/480 [==============================] - 100% - Took: 8.8 min - loss: 0.630 - acc: 0.681 - val_loss: 0.628 - val_acc: 0.681

Epoch 4/30
step: 480/480 [==============================] - 100% - Took: 7.7 min - loss: 0.629 - acc: 0.681 - val_loss: 0.627 - val_acc: 0.681

Epoch 5/30
step: 480/480 [==============================] - 100% - Took: 6.1 min - loss: 0.628 - acc: 0.681 - val_loss: 0.627 - val_acc: 0.681

Epoch 6/30
step: 480/480 [==============================] - 100% - Took: 6.0 min - loss: 0.628 - acc: 0.681 - val_loss: 0.627 - val_acc: 0.681

Epoch 7/30
step: 48

In [14]:
# Check what variables are available
print("Available variables:")
print([var for var in dir() if not var.startswith('_') and 'model' in var.lower()])

# Also check if you have any model-like objects
for var in dir():
    if not var.startswith('_'):
        try:
            obj = eval(var)
            if 'model' in str(type(obj)).lower():
                print(f"\n✅ Found model: {var}")
                print(f"   Type: {type(obj)}")
        except:
            pass

Available variables:
['Model']

✅ Found model: random_search
   Type: <class 'sklearn.model_selection._search.RandomizedSearchCV'>


In [1]:
print("="*60)
print(" XRNN MODEL EVALUATION")
print("="*60)

# Get predictions on test set
print("Generating predictions on test set...")
y_pred_xrnn = cnn_model.predict(X_test_cnn)
y_pred_xrnn_binary = (y_pred_xrnn > 0.5).astype(int).flatten()

# Get probability scores (if available, otherwise use predictions)
if hasattr(cnn_model, 'predict_proba'):
    y_prob_xrnn = cnn_model.predict_proba(X_test_cnn)[:, 1]
else:
    # Approximate probabilities from confidence
    y_prob_xrnn = np.abs(y_pred_xrnn - 0.5) * 2  # Simple confidence approximation

print(f" Test predictions complete! {len(y_test)} samples evaluated")

 XRNN MODEL EVALUATION
Generating predictions on test set...


NameError: name 'cnn_model' is not defined

In [2]:
# List all variables to find the xrNN model
print("Available variables:")
print([var for var in dir() if not var.startswith('_')])

# Check common xrNN model variable names
possible_names = ['cnn_model', 'model', 'xrnn_model', 'xrnn_cnn', 'trained_model']
for name in possible_names:
    if name in dir():
        var_value = eval(name)
        print(f"\n Found '{name}' - Type: {type(var_value)}")
        # If it's a model, try to use it
        if 'xrnn' in str(type(var_value)).lower() or 'model' in str(type(var_value)).lower():
            xrnn_model = var_value
            print(f"   Using '{name}' as xrnn_model")
            break
else:
    print("\n No xrnn model found in variables")
    
# If you remember what you named your model, check that specifically
# For example, if you named it 'my_model', add it to the list above

Available variables:
['In', 'Out', 'PS1', 'REPLHooks', 'exit', 'get_ipython', 'get_last_command', 'is_wsl', 'open', 'original_ps1', 'platform', 'quit', 'sys']

 No xrnn model found in variables


In [3]:
import os
print("Files in models/UNSW:")
if os.path.exists('../models/UNSW'):
    print(os.listdir('../models/UNSW'))
else:
    print("Folder doesn't exist")

Files in models/UNSW:
['best_pytorch_cnn.pth', 'best_rbf_params.pkl', 'best_svm_params.pkl', 'class_weights.pkl', 'encoders.pkl', 'final_svm_model.pkl', 'scaler.pkl']
